In [2]:
import pandas as pd

soil = pd.read_parquet('../data/bronze/soil_readings')
print("Shape:", soil.shape)
print("\nColumns:", soil.columns.tolist())
print("\nSample:\n", soil.head(3))


Shape: (5002099, 12)

Columns: ['Local_Time', 'Site_Name', 'Site_ID', 'ID', 'Probe_ID', 'Probe_Measure', 'Soil_Value', 'Unit', 'json_featuretype', 'event_time', 'ingested_at', 'ingested_date']

Sample:
            Local_Time                    Site_Name Site_ID        ID Probe_ID  \
0 2025-04-09 17:00:00           Royal Parade CSIRO   88403  26868350  2031673   
1 2025-04-09 18:00:00  Kings Domain South fireyard   66199  26868390  1338696   
2 2025-04-09 14:00:00     Princess bridge East p06  101038  26868410  2019364   

           Probe_Measure  Soil_Value   Unit json_featuretype  \
0     Soil Salinity 20cm        0.51  µS/cm           Output   
1  Soil Salinity 60cm #0        1.02  µS/cm           Output   
2  Soil Temperature 20cm       16.42     ºC           Output   

           event_time                ingested_at ingested_date  
0 2025-04-09 17:00:00 2026-04-08 11:43:11.057522    2026-04-08  
1 2025-04-09 18:00:00 2026-04-08 11:43:11.057522    2026-04-08  
2 2025-04-09 14:00:0

In [3]:
print("Unique sites:", soil['Site_Name'].nunique())
print("\nSite names:\n", soil['Site_Name'].value_counts())



Unique sites: 73

Site names:
 Site_Name
Royal Parade CSIRO                                214784
Princess bridge East p06                          212700
Royal Parade Opposite Lawn 5                      204704
Royal Parade South                                204032
Princess bridge SE P01                            202700
                                                   ...  
Bourke North 6                                      1152
Southbank Bvd Playspace                               96
Queen Vic (could not locate)                          96
Kings Domain West (could not locate)                  64
Kings Domain South (could not locate) fireyard        64
Name: count, Length: 73, dtype: int64


In [4]:
print(soil['Probe_Measure'].value_counts())


Probe_Measure
Soil Moisture 10cm #0       122763
Soil Moisture 20cm #0       122762
Soil Moisture 30cm #0       122748
Soil Moisture 40cm #0       122748
Soil Temperature 10cm #0    122633
                             ...  
Soil Moisture 070cm              4
Soil Temperature 050cm           4
Soil Moisture 060cm              4
Soil Temperature 080cm           4
Soil Temperature 070cm           4
Name: count, Length: 170, dtype: int64


In [5]:
import re

# Extract just the measurement type (Moisture, Temperature, Salinity)
soil['measure_type'] = soil['Probe_Measure'].str.extract(r'(Moisture|Temperature|Salinity|Conductivity)')
print(soil['measure_type'].value_counts())


measure_type
Moisture       2470277
Temperature    1428922
Salinity       1102900
Name: count, dtype: int64


In [6]:
soil['event_time'] = pd.to_datetime(soil['event_time'], utc=True)
print("Date range:", soil['event_time'].min(), "->", soil['event_time'].max())
print("\nReadings per month:")
print(soil.set_index('event_time').resample('ME').size())



Date range: 2023-09-02 10:30:00+00:00 -> 2026-01-11 21:15:00+00:00

Readings per month:
event_time
2023-09-30 00:00:00+00:00    106784
2023-10-31 00:00:00+00:00    250479
2023-11-30 00:00:00+00:00    272026
2023-12-31 00:00:00+00:00    227598
2024-01-31 00:00:00+00:00    218989
2024-02-29 00:00:00+00:00    204781
2024-03-31 00:00:00+00:00    207932
2024-04-30 00:00:00+00:00    203470
2024-05-31 00:00:00+00:00    198413
2024-06-30 00:00:00+00:00    204997
2024-07-31 00:00:00+00:00    196309
2024-08-31 00:00:00+00:00    198764
2024-09-30 00:00:00+00:00    176371
2024-10-31 00:00:00+00:00    174579
2024-11-30 00:00:00+00:00    162408
2024-12-31 00:00:00+00:00    160215
2025-01-31 00:00:00+00:00    151726
2025-02-28 00:00:00+00:00    136600
2025-03-31 00:00:00+00:00    142032
2025-04-30 00:00:00+00:00    146260
2025-05-31 00:00:00+00:00    154808
2025-06-30 00:00:00+00:00    123475
2025-07-31 00:00:00+00:00     78029
2025-08-31 00:00:00+00:00    151290
2025-09-30 00:00:00+00:00    176056
2

In [7]:
moisture = soil[soil['measure_type'] == 'Moisture']
print(moisture['Soil_Value'].describe())
print("\nNull values:", moisture['Soil_Value'].isna().sum())
print("Zero values:", (moisture['Soil_Value'] == 0).sum())
print("Negative values:", (moisture['Soil_Value'] < 0).sum())
print("Above 100%:", (moisture['Soil_Value'] > 100).sum())


count    2.470205e+06
mean     3.307910e+01
std      1.468228e+01
min     -2.118000e+01
25%      2.335000e+01
50%      3.188000e+01
75%      4.049000e+01
max      1.081600e+02
Name: Soil_Value, dtype: float64

Null values: 72
Zero values: 100
Negative values: 9704
Above 100%: 550


In [8]:
salinity = soil[soil['measure_type'] == 'Salinity']
print(salinity['Soil_Value'].describe())
print("\nNegative values:", (salinity['Soil_Value'] < 0).sum())
print("Suspiciously high (>10 µS/cm):", (salinity['Soil_Value'] > 10).sum())


count    1.102864e+06
mean     2.952839e-01
std      2.784840e-01
min      0.000000e+00
25%      1.000000e-01
50%      2.200000e-01
75%      3.900000e-01
max      2.160000e+00
Name: Soil_Value, dtype: float64

Negative values: 0
Suspiciously high (>10 µS/cm): 0


In [9]:
# What depths do we have?
print(soil['Probe_Measure'].value_counts().head(30))



Probe_Measure
Soil Moisture 10cm #0       122763
Soil Moisture 20cm #0       122762
Soil Moisture 30cm #0       122748
Soil Moisture 40cm #0       122748
Soil Temperature 10cm #0    122633
Soil Temperature 20cm #0    122631
Soil Temperature 30cm #0    122615
Soil Temperature 40cm #0    122615
Soil Salinity 20cm #0       105014
Soil Salinity 10cm #0       105014
Soil Salinity 30cm #0       104998
Soil Moisture 60cm #0       100755
Soil Moisture 70cm #0       100755
Soil Moisture 50cm #0       100755
Soil Moisture 80cm #0       100754
Soil Temperature 60cm #0    100612
Soil Temperature 50cm #0    100612
Soil Temperature 70cm #0    100612
Soil Temperature 80cm #0    100612
Soil Salinity 40cm #0        83047
Soil Salinity 60cm #0        83036
Soil Salinity 80cm #0        83036
Soil Salinity 70cm #0        83036
Soil Salinity 50cm #0        83036
Soil Temperature 20cm        64465
Soil Temperature 40cm        64465
Soil Moisture 10cm           64465
Soil Moisture 20cm           64465
Soil M

In [10]:
# Extract depth from probe measure name
import re

def extract_depth(probe_measure):
    match = re.search(r'(\d+)\s*cm', str(probe_measure), re.IGNORECASE)
    return int(match.group(1)) if match else None

soil['depth_cm'] = soil['Probe_Measure'].apply(extract_depth)
print("Unique depths:", sorted(soil['depth_cm'].dropna().unique()))
print("\nRecords per depth:")
print(soil['depth_cm'].value_counts().sort_index())


Unique depths: [np.float64(10.0), np.float64(20.0), np.float64(30.0), np.float64(40.0), np.float64(50.0), np.float64(60.0), np.float64(70.0), np.float64(80.0)]

Records per depth:
depth_cm
10.0    755250
20.0    756178
30.0    756118
40.0    756116
50.0    450273
60.0    450273
70.0    450273
80.0    450270
Name: count, dtype: int64


In [11]:
# Are #0 probes and non-#0 probes at the same sites?
has_hash = soil[soil['Probe_Measure'].str.contains('#0')]['Site_Name'].unique()
no_hash = soil[~soil['Probe_Measure'].str.contains('#0')]['Site_Name'].unique()

print("Sites with #0 probes:", len(has_hash))
print("Sites without #0:", len(no_hash))
print("Overlap:", len(set(has_hash) & set(no_hash)))
print("\nSites with NO #0 (only plain):", set(no_hash) - set(has_hash))


Sites with #0 probes: 61
Sites without #0: 12
Overlap: 0

Sites with NO #0 (only plain): {'Bourke South 6 (297)', 'Block 2', 'Block 3', 'Princess bridge SE P01', 'Block 1', 'Southbank Bvd Playspace', 'Princess bridge NW p01', 'Princess bridge SW p10', 'Princess bridge NE p11', 'Royal Parade Opposite Lawn 5', 'Royal Parade CSIRO', 'Princess bridge East p06'}


In [13]:
has_hash = soil[soil['Probe_Measure'].str.contains('#0')]['Site_Name'].unique()
for s in sorted(has_hash)[:20]:
    print(s)


5th Fairway
8th Green
9th Fairway
Alexandra Gardens Engineers Lawn
Alexandra Gardens Star Lawn
Argyle Square
Bandstand
Batman Park
Bourke North 1
Bourke North 2
Bourke North 3
Bourke North 4
Bourke North 5
Bourke North 6
Bourke North 6 (280)
Bourke South 1
Bourke South 2
Bourke South 4
Bourke South 5
Carlton Gardens North


In [14]:
temp = soil[soil['measure_type'] == 'Temperature']
print(temp['Soil_Value'].describe())
print("\nNegative values:", (temp['Soil_Value'] < 0).sum())
print("Suspiciously high (>60°C):", (temp['Soil_Value'] > 60).sum())


count    1.428562e+06
mean    -2.825304e+02
std      1.703365e+03
min     -9.999000e+03
25%      1.240000e+01
50%      1.573000e+01
75%      1.908000e+01
max      3.822000e+01
Name: Soil_Value, dtype: float64

Negative values: 42594
Suspiciously high (>60°C): 0


In [15]:
coverage = soil.groupby('Site_Name')['measure_type'].nunique()
print("Sites with all 3 measure types:", (coverage == 3).sum())
print("Sites with only 2:", (coverage == 2).sum())
print("Sites with only 1:", (coverage == 1).sum())


Sites with all 3 measure types: 63
Sites with only 2: 10
Sites with only 1: 0


In [16]:
depth_coverage = soil.groupby('Site_Name')['depth_cm'].nunique()
print(depth_coverage.value_counts().sort_index())


depth_cm
4    22
8    51
Name: count, dtype: int64


In [17]:
print("Soil date range:", soil['event_time'].min(), "→", soil['event_time'].max())

import pandas as pd
rainfall = pd.read_parquet('../data/bronze/weather_rainfall')
print("Rainfall date range:", rainfall['date'].min(), "→", rainfall['date'].max())

solar = pd.read_parquet('../data/bronze/weather_solar')
print("Solar date range:", solar['date'].min(), "→", solar['date'].max())

temperature = pd.read_parquet('../data/bronze/weather_temperature')
print("Temperature date range:", temperature['date'].min(), "→", temperature['date'].max())


Soil date range: 2023-09-02 10:30:00+00:00 → 2026-01-11 21:15:00+00:00
Rainfall date range: 2013-01-01 → 2025-12-17
Solar date range: 1990-01-01 → 2025-12-16
Temperature date range: 2013-01-01 → 2025-12-16


In [18]:
missing = soil.groupby('Site_Name')['measure_type'].nunique()
missing_sites = missing[missing < 3].index.tolist()
print(soil[soil['Site_Name'].isin(missing_sites)].groupby(['Site_Name','measure_type']).size().unstack(fill_value=0))


measure_type             Moisture  Temperature
Site_Name                                     
Bandstand                   14560         7280
Batman Park                 42528        21424
Block 1                     15712        63899
Conservatory                27656        13832
Darling Square              51720        26240
Fawkner North               28296        14192
Natures Play                51080        25576
Southbank Bvd Playspace        64           32
Speakers Corner             14992         7496
Treasury West               50592        25304


In [19]:
# Pick one site, one measure, one depth — look at time between readings
sample = soil[
    (soil['Site_Name'] == 'Batman Park') & 
    (soil['measure_type'] == 'Moisture') &
    (soil['depth_cm'] == 10)
].sort_values('event_time')

print(sample['event_time'].diff().describe())
print("\nFirst 5 timestamps:")
print(sample['event_time'].head())


count                         5315
mean     0 days 03:37:18.570084666
std      1 days 14:28:01.998007116
min                0 days 00:00:00
25%                0 days 00:00:00
50%                0 days 00:03:15
75%                0 days 02:00:00
max              113 days 19:00:00
Name: event_time, dtype: object

First 5 timestamps:
2425307   2023-11-01 15:00:00+00:00
1130088   2023-11-01 15:00:00+00:00
2134297   2023-11-01 17:00:00+00:00
1054315   2023-11-01 17:00:00+00:00
3190171   2023-11-02 11:00:00+00:00
Name: event_time, dtype: datetime64[ns, UTC]


In [20]:
# Check if Site_ID or any column gives us location info
print(soil[['Site_Name', 'Site_ID']].drop_duplicates().head(20))


                           Site_Name Site_ID
0                 Royal Parade CSIRO   88403
1        Kings Domain South fireyard   66199
2           Princess bridge East p06  101038
3                        Batman Park   75504
5                 Royal Parade South   72758
6   Alexandra Gardens Engineers Lawn   66195
7             Princess bridge SE P01  101036
8                     H.G.Smith Oval   64985
9               Bourke South 6 (297)   88425
12          Fitzroy Gardens West '18   65011
13   Royal Parade University college   70829
16         Fitzroy Gardens South '18   65007
18                           Block 3  193880
35                        Ryder Oval   64987
43                 Kings Domain West   66198
45            Princess bridge NE p11  101037
57          Fitzroy Gardens East '18   64990
58                    Bourke North 1   65016
59                         Queen Vic   66204
68              Bourke North 6 (280)   66202
